## Estudio resolucion temporal

In [10]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from IPython.display import display, HTML
import time
import pickle
import os
import sys
import csv
import time

In [18]:
def run_simulation(nx, ny, nt):
    # Parámetros adimensionales
    Re, Pr, Ec, Eu = 20.0, 10.0, 0.1, 1.0
    Lx_star, Ly_star = 1.0, 1.0
    dx_star = Lx_star / (nx - 1)
    dy_star = Ly_star / (ny - 1)
    up, u0 = -0.5, 1.0
    T0_star, T1_star = 0.0, 0.0
    dt_star = 1 / nt

    x_star = np.linspace(0, Lx_star, nx)
    y_star = np.linspace(0, Ly_star, ny)
    X_star, Y_star = np.meshgrid(x_star, y_star)

    u_star = np.zeros((ny, nx)) + u0
    v_star = np.zeros((ny, nx))
    p_star = np.zeros((ny, nx))
    T_star = np.zeros((ny, nx)) + T1_star

    # Condiciones de frontera iniciales
    u_star[0, :] = up
    u_star[-1, :] = u0
    v_star[0, :] = v_star[-1, :] = 0
    T_star[0, :] = T0_star
    T_star[-1, :] = T1_star

    u_history, v_history, p_history, T_history, tau_history = [], [], [], [], []

    for n in range(nt):
        u_old, v_old, p_old, T_old = u_star.copy(), v_star.copy(), p_star.copy(), T_star.copy()

        # Verificación CFL
        u_max = np.max(np.abs(u_old))
        v_max = np.max(np.abs(v_old))

        # Ajuste dinámico del dt_star para mantener CFL < 1
        cfl_max = 0.5
        dt_star = cfl_max * min(dx_star / (u_max + 1e-8), dy_star / (v_max + 1e-8))

        # Verificación CFL (ahora solo informativa)
        cfl_x = u_max * dt_star / dx_star
        cfl_y = v_max * dt_star / dy_star
        if cfl_x > 1.0 or cfl_y > 1.0:
            print(f"Advertencia: condición CFL violada en paso {n}: CFLx={cfl_x:.2f}, CFLy={cfl_y:.2f}")

        # Ecuaciones de momento
        for i in range(1, ny - 1):
            for j in range(1, nx - 1):
                conv_u_x = u_old[i, j] * (u_old[i, j + 1] - u_old[i, j - 1]) / (2 * dx_star)
                conv_u_y = v_old[i, j] * (u_old[i + 1, j] - u_old[i - 1, j]) / (2 * dy_star)
                diff_u = ((u_old[i, j + 1] - 2 * u_old[i, j] + u_old[i, j - 1]) / dx_star ** 2 +
                          (u_old[i + 1, j] - 2 * u_old[i, j] + u_old[i - 1, j]) / dy_star ** 2)

                u_star[i, j] = u_old[i, j] + dt_star * (-conv_u_x - conv_u_y + (1 / Re) * diff_u)

                conv_v_x = u_old[i, j] * (v_old[i, j + 1] - v_old[i, j - 1]) / (2 * dx_star)
                conv_v_y = v_old[i, j] * (v_old[i + 1, j] - v_old[i - 1, j]) / (2 * dy_star)
                diff_v = ((v_old[i, j + 1] - 2 * v_old[i, j] + v_old[i, j - 1]) / dx_star ** 2 +
                          (v_old[i + 1, j] - 2 * v_old[i, j] + v_old[i - 1, j]) / dy_star ** 2)
                grad_p_y = (p_old[i + 1, j] - p_old[i - 1, j]) / (2 * dy_star)

                v_star[i, j] = v_old[i, j] + dt_star * (-conv_v_x - conv_v_y - Eu * grad_p_y + (1 / Re) * diff_v)

        # Solver de presión con criterio de convergencia
        for _ in range(200):  # Iteraciones máximas
            max_diff = 0.0
            for i in range(1, ny - 1):
                for j in range(1, nx - 1):
                    rhs = ((u_star[i, j + 1] - u_star[i, j - 1]) / (2 * dx_star) +
                           (v_star[i + 1, j] - v_star[i - 1, j]) / (2 * dy_star))
                    new_p = 0.25 * (p_old[i + 1, j] + p_old[i - 1, j] + p_old[i, j + 1] + p_old[i, j - 1] -
                                    (dx_star * dy_star) / (2 * (dx_star ** 2 + dy_star ** 2)) * rhs)
                    max_diff = max(max_diff, abs(new_p - p_star[i, j]))
                    p_star[i, j] = new_p
            if max_diff < 1e-4:
                break

        # Corregir velocidades con presión
        for i in range(1, ny - 1):
            for j in range(1, nx - 1):
                u_star[i, j] -= dt_star * Eu * (p_star[i, j + 1] - p_star[i, j - 1]) / (2 * dx_star)
                v_star[i, j] -= dt_star * Eu * (p_star[i + 1, j] - p_star[i - 1, j]) / (2 * dy_star)

        # Ecuación de energía
        for i in range(1, ny - 1):
            for j in range(1, nx - 1):
                conv_T_x = u_star[i, j] * (T_old[i, j + 1] - T_old[i, j - 1]) / (2 * dx_star)
                conv_T_y = v_star[i, j] * (T_old[i + 1, j] - T_old[i - 1, j]) / (2 * dy_star)
                diff_T = ((T_old[i, j + 1] - 2 * T_old[i, j] + T_old[i, j - 1]) / dx_star ** 2 +
                          (T_old[i + 1, j] - 2 * T_old[i, j] + T_old[i - 1, j]) / dy_star ** 2)

                Sxx = (u_star[i, j + 1] - u_star[i, j - 1]) / (2 * dx_star)
                Syy = (v_star[i + 1, j] - v_star[i - 1, j]) / (2 * dy_star)
                Sxy = 0.5 * ((u_star[i + 1, j] - u_star[i - 1, j]) / (2 * dy_star) +
                             (v_star[i, j + 1] - v_star[i, j - 1]) / (2 * dx_star))
                viscous_heating = (Ec / Re) * (2 * (Sxx ** 2 + Syy ** 2) + 4 * Sxy ** 2)

                T_star[i, j] = T_old[i, j] + dt_star * (-conv_T_x - conv_T_y +
                                                       (1 / (Re * Pr)) * diff_T + viscous_heating)

        # Shear stress
        tau = np.zeros_like(u_star)
        tau[1:-1, :] = (u_star[2:, :] - u_star[:-2, :]) / (2 * dy_star)

        # Condiciones de frontera actualizadas
        u_star[0, :] = up
        u_star[-1, :] = u0
        u_star[:, -1] = u_star[:, -2]  # Neumann en x=Lx
        v_star[0, :] = v_star[-1, :] = 0
        v_star[:, -1] = 0

        p_star[:, 0] = p_star[:, 1]
        p_star[:, -1] = p_star[:, -2]
        p_star[0, :] = p_star[1, :]
        p_star[-1, :] = p_star[-2, :]

        if n % 10 == 0:
            print(f"\rSimulación en progreso: paso {n}/{nt}", end="")
            sys.stdout.flush()
            u_history.append(u_star.copy())
            v_history.append(v_star.copy())
            p_history.append(p_star.copy())
            T_history.append(T_star.copy())
            tau_history.append(tau.copy())
    print("\rSimulación finalizada.                      ")

    return {
        "u_history": u_history,
        "v_history": v_history,
        "p_history": p_history,
        "T_history": T_history,
        "tau_history": tau_history,
        "params": {"nx": nx, "ny": ny, "dt_star": dt_star, "nt": nt},
        "X_star": X_star,
        "Y_star": Y_star
    }


In [19]:
resoluciones = [(25, 25), (50, 50), (100, 100), (200, 200), (400, 400)]
nt_inicial = 1100
nt_incremento = 100
nt_max = 6000
tol = 1e-2

def variable_converge(history, tol):
    for i in range(1, len(history)):
        diff = np.max(np.abs(history[i] - history[i - 1]))
        if diff >= tol:
            return False
    return True

def encontrar_nt_convergencia(nx, ny):
    nt = nt_inicial
    while nt <= nt_max:
        print(f"\nEjecutando simulación para {nx}x{ny} con nt={nt}")
        resultado = run_simulation(nx, ny, nt)

        convergencias = {}
        for nombre, hist in [("u", resultado["u_history"]),
                             ("v", resultado["v_history"]),
                             ("p", resultado["p_history"]),
                             ("T", resultado["T_history"]),
                             ("tau", resultado["tau_history"])]:
            convergencias[nombre] = variable_converge(hist[-5:], tol)  # revisar últimas 5 muestras (~50 pasos)

        if all(convergencias.values()):
            return nt, convergencias
        nt += nt_incremento
    return None, convergencias

# Bucle principal
for nx, ny in resoluciones:
    nt_final, estados = encontrar_nt_convergencia(nx, ny)
    print(f"\n📊 Resolución {nx}x{ny}:")
    if nt_final:
        print(f"✔️ Convergencia completa alcanzada con nt = {nt_final}")
    else:
        print("❌ No se alcanzó convergencia completa antes de nt = 6000")
    for var, estado in estados.items():
        print(f"  - {var}: {'✅' if estado else '❌'}")



Ejecutando simulación para 25x25 con nt=1100
Simulación en progreso: paso 0/1100

Simulación finalizada.                      

Ejecutando simulación para 25x25 con nt=1200
Simulación finalizada.                      

Ejecutando simulación para 25x25 con nt=1300
Simulación finalizada.                      

Ejecutando simulación para 25x25 con nt=1400
Simulación en progreso: paso 1340/1400

KeyboardInterrupt: 

## Nuevo codigo para reescalado con 'garantia' de alcanzar el estado estacionario

In [20]:
def run_simulation(nx, ny):
    # Parámetros adimensionales
    Re, Pr, Ec, Eu = 20.0, 10.0, 0.1, 1.0
    Lx_star, Ly_star = 1.0, 1.0
    dx_star = Lx_star / (nx - 1)
    dy_star = Ly_star / (ny - 1)
    up, u0 = -0.5, 1.0
    T0_star, T1_star = 0.0, 0.0
    cfl_max = 0.5  # CFL máximo

    x_star = np.linspace(0, Lx_star, nx)
    y_star = np.linspace(0, Ly_star, ny)
    X_star, Y_star = np.meshgrid(x_star, y_star)

    u_star = np.zeros((ny, nx)) + u0
    v_star = np.zeros((ny, nx))
    p_star = np.zeros((ny, nx))
    T_star = np.zeros((ny, nx)) + T1_star

    # Condiciones de frontera iniciales
    u_star[0, :] = up
    u_star[-1, :] = u0
    v_star[0, :] = v_star[-1, :] = 0
    T_star[0, :] = T0_star
    T_star[-1, :] = T1_star

    u_history, v_history, p_history, T_history, tau_history = [], [], [], [], []

    t_star = 0.0
    step = 0

    while t_star < 1.0:
        u_old, v_old, p_old, T_old = u_star.copy(), v_star.copy(), p_star.copy(), T_star.copy()

        u_max = np.max(np.abs(u_old))
        v_max = np.max(np.abs(v_old))
        dt_star = cfl_max * min(dx_star / (u_max + 1e-8), dy_star / (v_max + 1e-8))

        # Ajustar el último paso si excede t*=1.0
        if t_star + dt_star > 1.0:
            dt_star = 1.0 - t_star

        t_star += dt_star

        # Ecuaciones de momento
        for i in range(1, ny - 1):
            for j in range(1, nx - 1):
                conv_u_x = u_old[i, j] * (u_old[i, j + 1] - u_old[i, j - 1]) / (2 * dx_star)
                conv_u_y = v_old[i, j] * (u_old[i + 1, j] - u_old[i - 1, j]) / (2 * dy_star)
                diff_u = ((u_old[i, j + 1] - 2 * u_old[i, j] + u_old[i, j - 1]) / dx_star ** 2 +
                          (u_old[i + 1, j] - 2 * u_old[i, j] + u_old[i - 1, j]) / dy_star ** 2)
                u_star[i, j] = u_old[i, j] + dt_star * (-conv_u_x - conv_u_y + (1 / Re) * diff_u)

                conv_v_x = u_old[i, j] * (v_old[i, j + 1] - v_old[i, j - 1]) / (2 * dx_star)
                conv_v_y = v_old[i, j] * (v_old[i + 1, j] - v_old[i - 1, j]) / (2 * dy_star)
                diff_v = ((v_old[i, j + 1] - 2 * v_old[i, j] + v_old[i, j - 1]) / dx_star ** 2 +
                          (v_old[i + 1, j] - 2 * v_old[i, j] + v_old[i - 1, j]) / dy_star ** 2)
                grad_p_y = (p_old[i + 1, j] - p_old[i - 1, j]) / (2 * dy_star)
                v_star[i, j] = v_old[i, j] + dt_star * (-conv_v_x - conv_v_y - Eu * grad_p_y + (1 / Re) * diff_v)

        # Solver de presión
        for _ in range(200):
            max_diff = 0.0
            for i in range(1, ny - 1):
                for j in range(1, nx - 1):
                    rhs = ((u_star[i, j + 1] - u_star[i, j - 1]) / (2 * dx_star) +
                           (v_star[i + 1, j] - v_star[i - 1, j]) / (2 * dy_star))
                    new_p = 0.25 * (p_old[i + 1, j] + p_old[i - 1, j] + p_old[i, j + 1] + p_old[i, j - 1] -
                                    (dx_star * dy_star) / (2 * (dx_star ** 2 + dy_star ** 2)) * rhs)
                    max_diff = max(max_diff, abs(new_p - p_star[i, j]))
                    p_star[i, j] = new_p
            if max_diff < 1e-4:
                break

        # Corregir velocidades
        for i in range(1, ny - 1):
            for j in range(1, nx - 1):
                u_star[i, j] -= dt_star * Eu * (p_star[i, j + 1] - p_star[i, j - 1]) / (2 * dx_star)
                v_star[i, j] -= dt_star * Eu * (p_star[i + 1, j] - p_star[i - 1, j]) / (2 * dy_star)

        # Energía
        for i in range(1, ny - 1):
            for j in range(1, nx - 1):
                conv_T_x = u_star[i, j] * (T_old[i, j + 1] - T_old[i, j - 1]) / (2 * dx_star)
                conv_T_y = v_star[i, j] * (T_old[i + 1, j] - T_old[i - 1, j]) / (2 * dy_star)
                diff_T = ((T_old[i, j + 1] - 2 * T_old[i, j] + T_old[i, j - 1]) / dx_star ** 2 +
                          (T_old[i + 1, j] - 2 * T_old[i, j] + T_old[i - 1, j]) / dy_star ** 2)

                Sxx = (u_star[i, j + 1] - u_star[i, j - 1]) / (2 * dx_star)
                Syy = (v_star[i + 1, j] - v_star[i - 1, j]) / (2 * dy_star)
                Sxy = 0.5 * ((u_star[i + 1, j] - u_star[i - 1, j]) / (2 * dy_star) +
                             (v_star[i, j + 1] - v_star[i, j - 1]) / (2 * dx_star))
                viscous_heating = (Ec / Re) * (2 * (Sxx ** 2 + Syy ** 2) + 4 * Sxy ** 2)

                T_star[i, j] = T_old[i, j] + dt_star * (-conv_T_x - conv_T_y +
                                                       (1 / (Re * Pr)) * diff_T + viscous_heating)

        # Shear stress
        tau = np.zeros_like(u_star)
        tau[1:-1, :] = (u_star[2:, :] - u_star[:-2, :]) / (2 * dy_star)

        # Condiciones de frontera
        u_star[0, :] = up
        u_star[-1, :] = u0
        u_star[:, -1] = u_star[:, -2]
        v_star[0, :] = v_star[-1, :] = 0
        v_star[:, -1] = 0
        p_star[:, 0] = p_star[:, 1]
        p_star[:, -1] = p_star[:, -2]
        p_star[0, :] = p_star[1, :]
        p_star[-1, :] = p_star[-2, :]

        if step % 10 == 0:
            print(f"\rSimulación en progreso: t*={t_star:.3f}", end="")
            sys.stdout.flush()
            u_history.append(u_star.copy())
            v_history.append(v_star.copy())
            p_history.append(p_star.copy())
            T_history.append(T_star.copy())
            tau_history.append(tau.copy())

        step += 1

    print("\rSimulación finalizada.                      ")

    return {
        "u_history": u_history,
        "v_history": v_history,
        "p_history": p_history,
        "T_history": T_history,
        "tau_history": tau_history,
        "params": {"nx": nx, "ny": ny, "Re": Re},
        "X_star": X_star,
        "Y_star": Y_star
    }


## Checkeo de equivalencia entre resoluciones espaciales

In [22]:
from scipy.interpolate import griddata
import numpy as np

# Ejecutar con 3 resoluciones
resolutions = [(21, 21), (41, 41), (81, 81), (161,161)]
results = [run_simulation(nx, ny) for nx, ny in resolutions]

# Definir malla de referencia (la más fina)
ref = results[-1]
x_ref, y_ref = ref["X_star"].flatten(), ref["Y_star"].flatten()
grid_ref = np.stack([x_ref, y_ref], axis=-1)

def interpolate_to_ref(field, X, Y):
    return griddata((X.flatten(), Y.flatten()), field.flatten(), grid_ref, method='cubic')

# Comparar campos
fields = ["u_history", "v_history", "p_history", "T_history", "tau_history"]
for i, res in enumerate(results[:-1]):
    print(f"\nComparando resolución {res['params']['nx']}x{res['params']['ny']} con referencia...")
    for key in fields:
        field_coarse = res[key][-1]
        Xc, Yc = res["X_star"], res["Y_star"]
        interp_field = interpolate_to_ref(field_coarse, Xc, Yc)
        field_ref = ref[key][-1].flatten()

        # Error relativo L2
        error = np.linalg.norm(interp_field - field_ref) / np.linalg.norm(field_ref)
        print(f"{key.replace('_history',''):>6} → Error relativo: {error:.2e}")


Simulación finalizada.                      
Simulación finalizada.                      
Simulación finalizada.                      
Simulación finalizada.                      

Comparando resolución 21x21 con referencia...
     u → Error relativo: 9.51e-01
     v → Error relativo: 9.87e-01
     p → Error relativo: 9.99e-01
     T → Error relativo: 1.00e+00
   tau → Error relativo: 9.97e-01

Comparando resolución 41x41 con referencia...
     u → Error relativo: 8.59e-01
     v → Error relativo: 9.30e-01
     p → Error relativo: 9.83e-01
     T → Error relativo: 1.00e+00
   tau → Error relativo: 9.62e-01

Comparando resolución 81x81 con referencia...
     u → Error relativo: 5.73e-01
     v → Error relativo: 7.62e-01
     p → Error relativo: 8.98e-01
     T → Error relativo: 1.00e+00
   tau → Error relativo: 8.75e-01
